# CRISP-DM Notebook: Predicting `orders.is_fraud`

This notebook implements the Chapter 17 assignment pipeline using `shop.db`.

![CRISP-DM Process](../crisp_dm_process.png)

## Business Understanding

- **Problem:** Detect potentially fraudulent orders before fulfillment.
- **Business objective:** Prioritize high-risk orders for review while minimizing false positives that block legitimate customers.
- **Primary metric:** ROC-AUC for ranking quality.
- **Operational metric:** Recall and precision at a chosen threshold for queueing flagged orders.

In [ ]:
import sqlite3
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [ ]:
# Data understanding: load from SQLite
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DB_PATH = ROOT / "shop.db"

conn = sqlite3.connect(DB_PATH)
customers = pd.read_sql_query("SELECT * FROM customers", conn)
orders = pd.read_sql_query("SELECT * FROM orders", conn)
shipments = pd.read_sql_query("SELECT * FROM shipments", conn)
order_items = pd.read_sql_query("SELECT * FROM order_items", conn)
products = pd.read_sql_query("SELECT * FROM products", conn)
conn.close()

print("customers", customers.shape)
print("orders", orders.shape)
print("shipments", shipments.shape)
print("order_items", order_items.shape)
print("products", products.shape)
orders.head()

In [ ]:
# Target exploration
fraud_rate = orders["is_fraud"].mean()
print(f"Fraud rate: {fraud_rate:.3f}")
orders["is_fraud"].value_counts().sort_index().plot(kind="bar", title="Class balance: is_fraud")
plt.show()

orders[["order_total", "risk_score", "is_fraud"]].groupby("is_fraud").agg(["mean", "median", "std"])

In [ ]:
# Data preparation + feature engineering
shipments_agg = shipments.groupby("order_id", as_index=False).agg(
    promised_days=("promised_days", "max"),
    actual_days=("actual_days", "max"),
    late_delivery=("late_delivery", "max"),
)

item_agg = order_items.groupby("order_id", as_index=False).agg(
    item_count=("order_item_id", "count"),
    quantity_sum=("quantity", "sum"),
)

df = (
    orders.merge(customers, on="customer_id", how="left")
    .merge(shipments_agg, on="order_id", how="left")
    .merge(item_agg, on="order_id", how="left")
)

for col in ["order_datetime", "birthdate", "created_at", "ship_datetime"]:
    if col in df.columns:
        dt = pd.to_datetime(df[col], errors="coerce")
        df[f"{col}_year"] = dt.dt.year
        df[f"{col}_month"] = dt.dt.month
        df[f"{col}_dow"] = dt.dt.dayofweek

if "birthdate" in df.columns:
    birth = pd.to_datetime(df["birthdate"], errors="coerce")
    order_dt = pd.to_datetime(df["order_datetime"], errors="coerce")
    df["customer_age"] = (order_dt - birth).dt.days / 365.25

if "actual_days" in df.columns and "promised_days" in df.columns:
    df["delivery_delay_days"] = df["actual_days"] - df["promised_days"]

target = "is_fraud"
exclude = {
    target,
    "order_id",
    "ship_datetime",
    "full_name",
    "email",
    "promo_code",
    "order_datetime",
    "birthdate",
    "created_at",
}
features = [c for c in df.columns if c not in exclude]

X = df[features].copy()
y = df[target].astype(int)

numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = [c for c in X.columns if c not in numeric_features]

print("Feature count:", len(features))
print("Numeric:", len(numeric_features), "Categorical:", len(categorical_features))
X.head()

In [ ]:
# Modeling: compare multiple classifiers
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]),
            numeric_features,
        ),
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore")),
            ]),
            categorical_features,
        ),
    ]
)

models = {
    "logistic": LogisticRegression(max_iter=1500, class_weight="balanced"),
    "random_forest": RandomForestClassifier(
        n_estimators=300, random_state=42, class_weight="balanced_subsample"
    ),
    "gradient_boosting": GradientBoostingClassifier(random_state=42),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = {}
for name, model in models.items():
    pipe = Pipeline([("prep", preprocessor), ("model", model)])
    scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="roc_auc", n_jobs=-1)
    cv_scores[name] = scores
    print(name, "ROC-AUC", round(scores.mean(), 4), "+/-", round(scores.std(), 4))

best_model_name = max(cv_scores, key=lambda k: cv_scores[k].mean())
best_model = Pipeline([("prep", preprocessor), ("model", models[best_model_name])])
best_model.fit(X_train, y_train)
print("Selected model:", best_model_name)

In [ ]:
# Evaluation + threshold tuning
y_proba = best_model.predict_proba(X_test)[:, 1]
roc = roc_auc_score(y_test, y_proba)

prec, rec, thresholds = precision_recall_curve(y_test, y_proba)
f1_vals = 2 * (prec * rec) / (prec + rec + 1e-9)
best_idx = int(np.nanargmax(f1_vals))
threshold = float(thresholds[max(best_idx - 1, 0)]) if len(thresholds) else 0.5

y_pred = (y_proba >= threshold).astype(int)

print("ROC-AUC:", round(roc, 4))
print("Threshold:", round(threshold, 4))
print("Precision:", round(precision_score(y_test, y_pred, zero_division=0), 4))
print("Recall:", round(recall_score(y_test, y_pred, zero_division=0), 4))
print("F1:", round(f1_score(y_test, y_pred, zero_division=0), 4))
print("\nConfusion matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification report:\n", classification_report(y_test, y_pred, zero_division=0))

In [ ]:
# Feature selection signal (tree-based importance on transformed matrix)
if best_model_name in {"random_forest", "gradient_boosting"}:
    transformed = best_model.named_steps["prep"].fit_transform(X_train)
    model = best_model.named_steps["model"]
    if hasattr(model, "feature_importances_"):
        importances = model.feature_importances_
        top_idx = np.argsort(importances)[::-1][:15]
        print("Top feature indices by importance:", top_idx)
        print("Top importance values:", np.round(importances[top_idx], 4))
else:
    print("Feature selection note: Logistic model selected; use coefficients/regularization for pruning.")

In [ ]:
# Deployment: serialize model artifact
artifact = {
    "pipeline": best_model,
    "threshold": threshold,
    "features": features,
    "model_name": best_model_name,
}

models_dir = ROOT / "models"
models_dir.mkdir(exist_ok=True)
out_file = models_dir / "fraud_model.joblib"
joblib.dump(artifact, out_file)
print("Saved:", out_file)

## Deployment Integration Note

The web app's scoring endpoint can load `models/fraud_model.joblib`, score newly created rows from `orders`, and write scored outputs to persistent tables (`scoring_runs`, `order_scores`) for warehouse queue display.

Minimal server-side sequence:
1. Read candidate orders from database.
2. Apply the saved preprocessing + model pipeline.
3. Store probabilities and run timestamp.
4. Refresh queue view using latest persisted scores.